# Dense Mapped Logit-KD: one-time Held-out Test

Notebook này thực hiện **một lần duy nhất** trên Karpathy Test:

- xác minh khóa phương pháp cuối và toàn bộ SHA-256;
- nạp đúng ba checkpoint đã khóa của seed `42`, `123`, `2026`;
- sinh caption cho đúng `5.000` ảnh bằng Beam-3, length penalty `0.7`;
- tính BLEU-1, BLEU-4, METEOR, ROUGE-L và CIDEr-D;
- báo cáo từng seed và `mean ± sample standard deviation`;
- không chọn checkpoint/seed, không chỉnh threshold và không thay đổi phương pháp theo Test.

Khi Cell 2 đọc caption Test, tập Test được xem là đã mở. Không dùng kết quả
của notebook này để huấn luyện lại hoặc thay đổi phương pháp.

In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import shutil
import statistics
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
assert INPUT_ROOT.is_dir(), "Notebook phải chạy trên Kaggle với các input bắt buộc."

RUN_NAME = "XLKD05_Dense_Mapped_LogitKD_held_out_test_one_time"
OUT = WORK_ROOT / RUN_NAME
AUDITS = OUT / "audits"
CONFIGS = OUT / "configs"
RESULTS = OUT / "results"
PREDICTIONS = OUT / "predictions"
MANIFESTS = OUT / "manifests"
for directory in (OUT, AUDITS, CONFIGS, RESULTS, PREDICTIONS, MANIFESTS):
    directory.mkdir(parents=True, exist_ok=True)

COMPLETE_MARKER = OUT / "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPLETE.json"
if COMPLETE_MARKER.exists():
    raise RuntimeError(
        "Held-out Test đã hoàn tất trong working directory này. "
        "Không được chạy lại để lựa chọn kết quả."
    )

def now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
    temporary.replace(path)

def candidates_by_name(name):
    return sorted(
        (path for path in INPUT_ROOT.rglob(name) if path.is_file()),
        key=lambda path: str(path),
    )

def exactly_one_name(name):
    matches = candidates_by_name(name)
    assert len(matches) == 1, (
        f"Cần đúng 1 file {name!r}, tìm thấy {len(matches)}: "
        + ", ".join(str(path) for path in matches[:10])
    )
    return matches[0]

def by_name_and_hash(name, expected_sha256):
    matches = [
        path for path in candidates_by_name(name)
        if sha256(path) == expected_sha256
    ]
    assert matches, (
        f"Không tìm thấy file {name!r} có SHA-256 {expected_sha256}."
    )
    if len(matches) > 1:
        print(
            f"INFO: {name} có {len(matches)} bản sao đồng nhất; "
            f"dùng {matches[0]}."
        )
    return matches[0]

def resolve_locked_artifact(marker_root, reference):
    relative = Path(reference["relative_path"])
    direct = marker_root / relative
    if direct.is_file() and sha256(direct) == reference["sha256"]:
        return direct
    return by_name_and_hash(relative.name, reference["sha256"])

marker_path = exactly_one_name(
    "XLKD05_Dense_Mapped_LogitKD_FINAL_METHOD_LOCKED.json"
)
marker = read_json(marker_path)
assert marker["status"] == "locked"
assert marker["selected_method"] == "cross-vocabulary dense mapped Logit-KD"
assert marker["paired_bootstrap_gate"] == "passed_3_of_3"
assert marker["held_out_test_protocol_authorized"] is True
assert marker["test_set_status"] == "sealed"
assert marker["test_set_used"] is False

contract_path = resolve_locked_artifact(marker_path.parent, marker["contract"])
audit_path = resolve_locked_artifact(marker_path.parent, marker["audit"])
manifest_path = resolve_locked_artifact(marker_path.parent, marker["manifest"])
contract = read_json(contract_path)
lock_audit = read_json(audit_path)

assert contract["status"] == "locked"
assert contract["method_id"] == "DENSE_MAPPED_LOGIT_KD"
assert contract["confirmation_seeds"] == [42, 123, 2026]
assert contract["held_out_test_protocol"]["open_count"] == 1
assert contract["held_out_test_protocol"]["evaluate_all_locked_seeds"] is True
assert contract["held_out_test_protocol"]["checkpoint_or_seed_selection_on_test"] is False
assert contract["held_out_test_protocol"]["threshold_tuning_on_test"] is False
assert contract["posthoc_method_changes_allowed"] is False
assert lock_audit["status"] == "passed"
assert lock_audit["test_set_status"] == "sealed"
assert lock_audit["test_set_used"] is False

DATASET_COCO_SHA256 = "af76e819987c9ced8452fdb4da823524e033f7baa097cf40bab83bdab4c469ea"
VOCAB_SHA256 = "1c6c7e6146d2700932c82be6f076d93153c757c10a1844076abaea9c11496f97"
DECODER_CONTRACT_SHA256 = "46655293c4a64506ea842dd0389715fbf64d19ae389c3ba79d2997fe5c3d62dc"

DATASET_COCO = by_name_and_hash("dataset_coco.json", DATASET_COCO_SHA256)
VOCAB_PATH = by_name_and_hash("word_vocabulary_minfreq5.json", VOCAB_SHA256)
DECODER_CONTRACT_PATH = by_name_and_hash(
    "shared_transformer_decoder_contract.json", DECODER_CONTRACT_SHA256
)

checkpoint_paths = {}
for record in contract["checkpoints"]:
    seed = int(record["seed"])
    name = Path(record["relative_to_run"]).name
    checkpoint_paths[seed] = by_name_and_hash(name, record["sha256"])
assert sorted(checkpoint_paths) == [42, 123, 2026]

preflight = {
    "schema_version": "held_out_test_preflight_v1",
    "created_at_utc": now(),
    "status": "passed",
    "method_lock": {
        "path": str(marker_path),
        "sha256": sha256(marker_path),
    },
    "contract": {"path": str(contract_path), "sha256": sha256(contract_path)},
    "lock_audit": {"path": str(audit_path), "sha256": sha256(audit_path)},
    "lock_manifest": {"path": str(manifest_path), "sha256": sha256(manifest_path)},
    "dataset_coco": {"path": str(DATASET_COCO), "sha256": sha256(DATASET_COCO)},
    "vocabulary": {"path": str(VOCAB_PATH), "sha256": sha256(VOCAB_PATH)},
    "decoder_contract": {
        "path": str(DECODER_CONTRACT_PATH),
        "sha256": sha256(DECODER_CONTRACT_PATH),
    },
    "checkpoints": {
        str(seed): {"path": str(path), "sha256": sha256(path)}
        for seed, path in checkpoint_paths.items()
    },
    "test_set_status": "sealed",
    "test_set_used": False,
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
    },
}
write_json(AUDITS / "held_out_test_cell01_preflight.json", preflight)

print("Preflight                 : PASSED")
print("Final method              :", contract["method"])
print("Locked seeds              :", sorted(checkpoint_paths))
print("Test state                : SEALED / NOT USED")
print("Next cell                 : one-time Test opening")

Preflight                 : PASSED
Final method              : cross-vocabulary dense mapped Logit-KD
Locked seeds              : [42, 123, 2026]
Test state                : SEALED / NOT USED
Next cell                 : one-time Test opening


In [2]:
from collections import defaultdict

from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

assert torch.cuda.is_available(), "Bắt buộc bật Kaggle GPU."
device = torch.device("cuda")
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

def find_image_leaf(directory_name, filename_prefix):
    matches = []
    for path in INPUT_ROOT.rglob(directory_name):
        if path.is_dir() and next(path.glob(f"{filename_prefix}*.jpg"), None):
            matches.append(path)
    matches = sorted(set(matches), key=lambda path: str(path))
    assert len(matches) == 1, (
        f"Cần đúng 1 thư mục ảnh {directory_name}, tìm thấy {len(matches)}: "
        + ", ".join(str(path) for path in matches[:10])
    )
    return matches[0]

IMAGE_LEAVES = {
    "train2014": find_image_leaf("train2014", "COCO_train2014_"),
    "val2014": find_image_leaf("val2014", "COCO_val2014_"),
}

# Đây là điểm mở Held-out Test duy nhất.
with DATASET_COCO.open("r", encoding="utf-8") as handle:
    karpathy = json.load(handle)
test_rows = [row for row in karpathy["images"] if row["split"] == "test"]
validation_ids = {
    int(row["cocoid"]) for row in karpathy["images"] if row["split"] == "val"
}
assert len(test_rows) == 5000

TEST_PATHS = {}
TEST_REFERENCES = defaultdict(list)
reference_count = 0
for row in test_rows:
    image_id = int(row["cocoid"])
    assert image_id not in TEST_PATHS
    partition = row["filepath"]
    assert partition in IMAGE_LEAVES
    image_path = IMAGE_LEAVES[partition] / row["filename"]
    assert image_path.is_file(), image_path
    TEST_PATHS[image_id] = image_path
    for sentence in row["sentences"]:
        caption = str(sentence["raw"]).strip()
        assert caption
        TEST_REFERENCES[image_id].append(caption)
        reference_count += 1

assert len(TEST_PATHS) == len(TEST_REFERENCES) == 5000
assert reference_count == 25010
assert set(TEST_PATHS).isdisjoint(validation_ids)
TEST_IMAGE_IDS = tuple(sorted(TEST_PATHS))

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def test_transform(image):
    image = TF.resize(
        image, 256, interpolation=InterpolationMode.BICUBIC, antialias=True
    )
    tensor = TF.to_tensor(TF.center_crop(image, [224, 224]))
    return TF.normalize(tensor, IMAGENET_MEAN, IMAGENET_STD)

class HeldOutTestImageDataset(Dataset):
    def __len__(self):
        return len(TEST_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = TEST_IMAGE_IDS[index]
        with Image.open(TEST_PATHS[image_id]) as source:
            image = test_transform(source.convert("RGB"))
        return image_id, image

def image_collate(rows):
    return {
        "image_ids": torch.tensor([row[0] for row in rows], dtype=torch.long),
        "student_images": torch.stack([row[1] for row in rows]),
    }

TEST_LOADER = DataLoader(
    HeldOutTestImageDataset(),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=image_collate,
)

references_path = RESULTS / "karpathy_held_out_test_references.json"
write_json(
    references_path,
    {str(image_id): TEST_REFERENCES[image_id] for image_id in TEST_IMAGE_IDS},
)

opening_state = {
    "schema_version": "held_out_test_opening_state_v1",
    "created_at_utc": now(),
    "opening_id": str(uuid.uuid4()),
    "open_count": 1,
    "authorization_marker_sha256": sha256(marker_path),
    "method_contract_sha256": sha256(contract_path),
    "karpathy_source_sha256": sha256(DATASET_COCO),
    "test_images": len(TEST_IMAGE_IDS),
    "test_references": reference_count,
    "validation_overlap_images": 0,
    "test_set_status": "opened_for_authorized_one_time_evaluation",
    "test_set_used": True,
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
}
write_json(OUT / "XLKD05_Dense_Mapped_LogitKD_TEST_OPENED.json", opening_state)
write_json(AUDITS / "held_out_test_cell02_data.json", opening_state)

print("Held-out Test             : OPENED (authorized one-time evaluation)")
print("Test images               :", len(TEST_IMAGE_IDS))
print("Reference captions        :", reference_count)
print("Validation overlap        : 0")

Held-out Test             : OPENED (authorized one-time evaluation)
Test images               : 5000
Reference captions        : 25010
Validation overlap        : 0


In [3]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_large

decoder_config = read_json(DECODER_CONTRACT_PATH)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["maximum_decoder_steps"]) == 31

class S1Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(torch.empty(1, 49, 256))
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens) + self.visual_position_embedding
        )

class SharedDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocab = int(config["vocabulary_size"])
        self.d_model = int(config["d_model"])
        self.maximum_steps = int(config["maximum_decoder_steps"])
        self.token_embedding = nn.Embedding(
            self.vocab, self.d_model, padding_idx=0
        )
        self.position_embedding = nn.Embedding(
            self.maximum_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(float(config["dropout"]))
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(torch.zeros(self.vocab))

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length, length, dtype=torch.bool, device=ids.device
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(hidden, self.token_embedding.weight, self.output_bias)

STUDENT_ENCODER = S1Encoder().to(device).eval()
STUDENT_DECODER = SharedDecoder(decoder_config).to(device).eval()
assert sum(parameter.numel() for parameter in STUDENT_ENCODER.parameters()) == 3231024
assert sum(parameter.numel() for parameter in STUDENT_DECODER.parameters()) == 5759328

vocabulary_payload = read_json(VOCAB_PATH)
if "id_to_token" in vocabulary_payload:
    raw_id_to_token = vocabulary_payload["id_to_token"]
    if isinstance(raw_id_to_token, list):
        ID_TO_TOKEN = {
            index: token for index, token in enumerate(raw_id_to_token)
        }
    else:
        ID_TO_TOKEN = {
            int(key): value for key, value in raw_id_to_token.items()
        }
else:
    ID_TO_TOKEN = {
        int(value): key
        for key, value in vocabulary_payload["token_to_id"].items()
    }
assert len(ID_TO_TOKEN) == 10080

PAD_ID, BOS_ID, EOS_ID, UNK_ID = 0, 1, 2, 3
VOCABULARY_SIZE = 10080
BEAM_SIZE = int(contract["beam_size"])
LENGTH_PENALTY = float(contract["length_penalty"])
MINIMUM_LEXICAL_TOKENS = int(contract["minimum_words"])
MAXIMUM_DECODER_STEPS = int(contract["maximum_new_tokens"])
FORBIDDEN_GENERATION_TOKEN_IDS = (PAD_ID, BOS_ID, UNK_ID)
assert (BEAM_SIZE, LENGTH_PENALTY, MINIMUM_LEXICAL_TOKENS) == (3, 0.7, 2)

def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == EOS_ID:
            break
        if token_id in FORBIDDEN_GENERATION_TOKEN_IDS:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)

def batched_beam_search(visual_memory):
    batch_size, memory_length, memory_dimension = visual_memory.shape
    assert (memory_length, memory_dimension) == (49, 256)
    beam_tokens = torch.full(
        (batch_size, BEAM_SIZE, 1),
        BOS_ID,
        dtype=torch.long,
        device=device,
    )
    beam_scores = torch.full(
        (batch_size, BEAM_SIZE),
        float("-inf"),
        dtype=torch.float32,
        device=device,
    )
    beam_scores[:, 0] = 0.0
    beam_finished = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.bool, device=device
    )
    beam_lengths = torch.zeros(
        (batch_size, BEAM_SIZE), dtype=torch.long, device=device
    )
    expanded_memory = (
        visual_memory.unsqueeze(1)
        .expand(-1, BEAM_SIZE, -1, -1)
        .reshape(batch_size * BEAM_SIZE, memory_length, memory_dimension)
        .contiguous()
    )

    for generation_step in range(MAXIMUM_DECODER_STEPS):
        flat_tokens = beam_tokens.reshape(batch_size * BEAM_SIZE, -1)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            logits = STUDENT_DECODER(
                flat_tokens, expanded_memory, flat_tokens.ne(PAD_ID)
            )
        log_probs = F.log_softmax(
            logits[:, -1, :].float(), dim=-1
        ).reshape(batch_size, BEAM_SIZE, VOCABULARY_SIZE)
        log_probs[:, :, list(FORBIDDEN_GENERATION_TOKEN_IDS)] = float("-inf")
        if generation_step < MINIMUM_LEXICAL_TOKENS:
            log_probs[:, :, EOS_ID] = float("-inf")
        if beam_finished.any():
            log_probs = log_probs.masked_fill(
                beam_finished.unsqueeze(-1), float("-inf")
            )
            finished_indices = beam_finished.nonzero(as_tuple=False)
            log_probs[
                finished_indices[:, 0], finished_indices[:, 1], EOS_ID
            ] = 0.0
        candidate_scores = beam_scores.unsqueeze(-1) + log_probs
        next_scores, flat_indices = torch.topk(
            candidate_scores.reshape(
                batch_size, BEAM_SIZE * VOCABULARY_SIZE
            ),
            k=BEAM_SIZE,
            dim=-1,
        )
        parent = flat_indices // VOCABULARY_SIZE
        next_tokens = flat_indices % VOCABULARY_SIZE
        parent_tokens = torch.gather(
            beam_tokens,
            1,
            parent.unsqueeze(-1).expand(-1, -1, beam_tokens.shape[-1]),
        )
        parent_finished = torch.gather(beam_finished, 1, parent)
        parent_lengths = torch.gather(beam_lengths, 1, parent)
        beam_tokens = torch.cat(
            [parent_tokens, next_tokens.unsqueeze(-1)], dim=-1
        )
        beam_scores = next_scores
        beam_lengths = parent_lengths + (~parent_finished).long()
        beam_finished = parent_finished | (next_tokens == EOS_ID)
        if bool(beam_finished.all()):
            break
    normalized = beam_scores / (
        beam_lengths.clamp_min(1).float() ** LENGTH_PENALTY
    )
    best = normalized.argmax(dim=1)
    batch_indices = torch.arange(batch_size, device=device)
    return {
        "token_ids": beam_tokens[batch_indices, best].cpu(),
        "raw_scores": beam_scores[batch_indices, best].cpu(),
        "normalized_scores": normalized[batch_indices, best].cpu(),
    }

def load_locked_checkpoint(seed):
    payload = torch.load(
        checkpoint_paths[seed], map_location="cpu", weights_only=False
    )
    assert payload["schema_version"] in {
        "xlkd04_dense_mapped_checkpoint_v1",
        "xlkd05_dense_mapped_checkpoint_v1",
    }
    assert payload["experiment_id"] == "XLKD_EXPANSIONNET_S1_DENSE_MAPPED_LOGITKD"
    assert int(payload["seed"]) == seed
    assert payload["test_set_status"] == "sealed"
    assert payload["test_set_used"] is False
    STUDENT_ENCODER.load_state_dict(payload["student_encoder_state"], strict=True)
    STUDENT_DECODER.load_state_dict(payload["student_decoder_state"], strict=True)
    STUDENT_ENCODER.eval()
    STUDENT_DECODER.eval()
    return payload

@torch.inference_mode()
def generate_test_captions(seed):
    load_locked_checkpoint(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    hypotheses = {}
    records = {}
    started = time.perf_counter()
    for batch_index, batch in enumerate(TEST_LOADER, start=1):
        image_ids = [int(value) for value in batch["image_ids"].tolist()]
        images = batch["student_images"].to(device, non_blocking=True)
        with torch.amp.autocast("cuda", dtype=torch.float16):
            memory = STUDENT_ENCODER(images)
        beam = batched_beam_search(memory)
        for row_index, image_id in enumerate(image_ids):
            token_ids = [
                int(value)
                for value in beam["token_ids"][row_index].tolist()
            ]
            caption = decode_tokens(token_ids)
            assert caption and image_id not in hypotheses
            hypotheses[image_id] = caption
            records[image_id] = {
                "image_id": image_id,
                "caption": caption,
                "token_ids": token_ids,
                "raw_score": float(beam["raw_scores"][row_index]),
                "normalized_score": float(
                    beam["normalized_scores"][row_index]
                ),
            }
        if batch_index % 25 == 0 or len(hypotheses) == len(TEST_IMAGE_IDS):
            print(
                f"seed={seed} | images={len(hypotheses):04d}/5000 | "
                f"elapsed={time.perf_counter() - started:.1f}s",
                flush=True,
            )
    assert set(hypotheses) == set(TEST_REFERENCES)
    return hypotheses, records, time.perf_counter() - started

generated = {}
generation_seconds = {}
for seed in (42, 123, 2026):
    prediction_path = PREDICTIONS / f"held_out_test_seed{seed}_predictions.json"
    hypotheses, records, elapsed = generate_test_captions(seed)
    payload = {
        "schema_version": "held_out_test_predictions_v1",
        "created_at_utc": now(),
        "method": contract["method"],
        "seed": seed,
        "checkpoint_sha256": sha256(checkpoint_paths[seed]),
        "split": "held_out_test",
        "images": len(records),
        "beam_size": BEAM_SIZE,
        "length_penalty": LENGTH_PENALTY,
        "minimum_words": MINIMUM_LEXICAL_TOKENS,
        "maximum_new_tokens": MAXIMUM_DECODER_STEPS,
        "generation_seconds": elapsed,
        "predictions": {
            str(image_id): records[image_id] for image_id in TEST_IMAGE_IDS
        },
        "test_set_status": "used_for_authorized_one_time_evaluation",
        "test_set_used": True,
    }
    write_json(prediction_path, payload)
    generated[seed] = hypotheses
    generation_seconds[seed] = elapsed

generation_audit = {
    "schema_version": "held_out_test_generation_audit_v1",
    "created_at_utc": now(),
    "status": "passed",
    "seeds": [42, 123, 2026],
    "images_per_seed": 5000,
    "generation_seconds": {
        str(seed): generation_seconds[seed] for seed in generation_seconds
    },
    "beam_size": BEAM_SIZE,
    "length_penalty": LENGTH_PENALTY,
    "test_set_status": "used_for_authorized_one_time_evaluation",
    "test_set_used": True,
}
write_json(AUDITS / "held_out_test_cell03_generation.json", generation_audit)
print("Generation               : PASSED 3/3 seeds")

seed=42 | images=0800/5000 | elapsed=189.7s
seed=42 | images=1600/5000 | elapsed=353.4s
seed=42 | images=2400/5000 | elapsed=524.8s
seed=42 | images=3200/5000 | elapsed=692.1s
seed=42 | images=4000/5000 | elapsed=859.6s
seed=42 | images=4800/5000 | elapsed=1031.1s
seed=42 | images=5000/5000 | elapsed=1073.9s
seed=123 | images=0800/5000 | elapsed=5.4s
seed=123 | images=1600/5000 | elapsed=10.2s
seed=123 | images=2400/5000 | elapsed=15.0s
seed=123 | images=3200/5000 | elapsed=19.9s
seed=123 | images=4000/5000 | elapsed=24.8s
seed=123 | images=4800/5000 | elapsed=29.6s
seed=123 | images=5000/5000 | elapsed=30.6s
seed=2026 | images=0800/5000 | elapsed=5.3s
seed=2026 | images=1600/5000 | elapsed=10.1s
seed=2026 | images=2400/5000 | elapsed=15.1s
seed=2026 | images=3200/5000 | elapsed=20.1s
seed=2026 | images=4000/5000 | elapsed=25.2s
seed=2026 | images=4800/5000 | elapsed=30.2s
seed=2026 | images=5000/5000 | elapsed=31.4s
Generation               : PASSED 3/3 seeds


In [4]:
try:
    from pycocoevalcap.bleu.bleu import Bleu
except ImportError:
    wheels = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl"),
        key=lambda path: str(path),
    )
    assert len(wheels) == 1, (
        "Thiếu offline wheel pycocoevalcap-1.2-py3-none-any.whl."
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-deps", str(wheels[0])]
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [{"caption": caption} for caption in captions]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(tokenized_gt, tokenized_pred)
    meteor, _ = Meteor().compute_score(tokenized_gt, tokenized_pred)
    rouge, _ = Rouge().compute_score(tokenized_gt, tokenized_pred)
    cider, _ = Cider().compute_score(tokenized_gt, tokenized_pred)
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }

metric_names = ("BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr-D")
per_seed_metrics = {}
for seed in (42, 123, 2026):
    metrics = compute_coco_metrics(TEST_REFERENCES, generated[seed])
    assert set(metrics) == set(metric_names)
    assert all(math.isfinite(value) for value in metrics.values())
    per_seed_metrics[seed] = metrics
    write_json(
        RESULTS / f"held_out_test_seed{seed}_metrics.json",
        {
            "schema_version": "held_out_test_seed_metrics_v1",
            "created_at_utc": now(),
            "method": contract["method"],
            "seed": seed,
            "split": "held_out_test",
            "images": 5000,
            "references": 25010,
            "metrics": metrics,
            "checkpoint_or_seed_selection_on_test": False,
            "threshold_tuning_on_test": False,
            "test_set_status": "used_for_authorized_one_time_evaluation",
            "test_set_used": True,
        },
    )

aggregate = {}
for metric in metric_names:
    values = [per_seed_metrics[seed][metric] for seed in (42, 123, 2026)]
    aggregate[metric] = {
        "mean": statistics.fmean(values),
        "sample_std": statistics.stdev(values),
        "values_by_seed": {
            str(seed): per_seed_metrics[seed][metric]
            for seed in (42, 123, 2026)
        },
    }

summary_path = RESULTS / "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_SUMMARY.json"
summary = {
    "schema_version": "dense_mapped_logit_kd_held_out_test_summary_v1",
    "created_at_utc": now(),
    "status": "complete",
    "method": contract["method"],
    "seeds": [42, 123, 2026],
    "split": "held_out_test",
    "images": 5000,
    "references": 25010,
    "per_seed_metrics": {
        str(seed): per_seed_metrics[seed] for seed in (42, 123, 2026)
    },
    "aggregate_mean_and_sample_std": aggregate,
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
    "test_open_count": 1,
    "test_set_status": "used_for_authorized_one_time_evaluation",
    "test_set_used": True,
}
write_json(summary_path, summary)

csv_path = RESULTS / "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_METRICS.csv"
with csv_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["seed", *metric_names],
    )
    writer.writeheader()
    for seed in (42, 123, 2026):
        writer.writerow({"seed": seed, **per_seed_metrics[seed]})
    writer.writerow(
        {
            "seed": "mean",
            **{metric: aggregate[metric]["mean"] for metric in metric_names},
        }
    )
    writer.writerow(
        {
            "seed": "sample_std",
            **{
                metric: aggregate[metric]["sample_std"]
                for metric in metric_names
            },
        }
    )

report_path = RESULTS / "XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_REPORT.md"
report_lines = [
    "# Dense Mapped Logit-KD — Held-out Test",
    "",
    "| Seed | BLEU-1 | BLEU-4 | METEOR | ROUGE-L | CIDEr-D |",
    "|---:|---:|---:|---:|---:|---:|",
]
for seed in (42, 123, 2026):
    row = per_seed_metrics[seed]
    report_lines.append(
        f"| {seed} | {row['BLEU-1']:.6f} | {row['BLEU-4']:.6f} | "
        f"{row['METEOR']:.6f} | {row['ROUGE-L']:.6f} | "
        f"{row['CIDEr-D']:.6f} |"
    )
report_lines.extend(["", "## Tổng hợp mean ± sample standard deviation", ""])
for metric in metric_names:
    report_lines.append(
        f"- {metric}: {aggregate[metric]['mean']:.6f} ± "
        f"{aggregate[metric]['sample_std']:.6f}"
    )
report_lines.extend(
    [
        "",
        "Test được mở đúng một lần sau khi phương pháp đã khóa. "
        "Không lựa chọn seed/checkpoint và không tinh chỉnh theo Test.",
    ]
)
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

artifact_paths = [
    references_path,
    *[
        PREDICTIONS / f"held_out_test_seed{seed}_predictions.json"
        for seed in (42, 123, 2026)
    ],
    *[
        RESULTS / f"held_out_test_seed{seed}_metrics.json"
        for seed in (42, 123, 2026)
    ],
    summary_path,
    csv_path,
    report_path,
    AUDITS / "held_out_test_cell01_preflight.json",
    AUDITS / "held_out_test_cell02_data.json",
    AUDITS / "held_out_test_cell03_generation.json",
    OUT / "XLKD05_Dense_Mapped_LogitKD_TEST_OPENED.json",
]
manifest_csv = MANIFESTS / "held_out_test_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "bytes", "sha256"]
    )
    writer.writeheader()
    for path in sorted(artifact_paths, key=lambda item: str(item)):
        writer.writerow(
            {
                "relative_path": path.relative_to(OUT).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": sha256(path),
            }
        )

complete = {
    "schema_version": "dense_mapped_logit_kd_held_out_test_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "method": contract["method"],
    "authorization_marker_sha256": sha256(marker_path),
    "method_contract_sha256": sha256(contract_path),
    "seeds": [42, 123, 2026],
    "test_images": 5000,
    "test_references": 25010,
    "summary": {
        "relative_path": summary_path.relative_to(OUT).as_posix(),
        "sha256": sha256(summary_path),
    },
    "metrics_csv": {
        "relative_path": csv_path.relative_to(OUT).as_posix(),
        "sha256": sha256(csv_path),
    },
    "report": {
        "relative_path": report_path.relative_to(OUT).as_posix(),
        "sha256": sha256(report_path),
    },
    "manifest": {
        "relative_path": manifest_csv.relative_to(OUT).as_posix(),
        "sha256": sha256(manifest_csv),
        "rows": len(artifact_paths),
    },
    "checkpoint_or_seed_selection_on_test": False,
    "threshold_tuning_on_test": False,
    "posthoc_method_changes": False,
    "test_open_count": 1,
    "test_set_status": "used_for_authorized_one_time_evaluation",
    "test_set_used": True,
}
write_json(COMPLETE_MARKER, complete)

print()
print("Held-out Test             : COMPLETE")
for metric in metric_names:
    print(
        f"{metric:24s}: {aggregate[metric]['mean']:.6f} "
        f"± {aggregate[metric]['sample_std']:.6f}"
    )
print("Test open count           : 1")
print("Seed/checkpoint selection : False")
print("Final marker              :", COMPLETE_MARKER)

Processing /kaggle/input/datasets/nguynthnhdan/dense-mapped-logitkd-held-out-test-static-input/XLKD05_Dense_Mapped_LogitKD_Held_Out_Test_Static_Input/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 307085 tokens at 545156.92 tokens per second.
PTBTokenizer tokenized 52231 tokens at 301431.40 tokens per second.


{'testlen': 47232, 'reflen': 47173, 'guess': [47232, 42232, 37232, 32232], 'correct': [35720, 19763, 9803, 4685]}
ratio: 1.0012507154516144


PTBTokenizer tokenized 307085 tokens at 676624.08 tokens per second.
PTBTokenizer tokenized 52071 tokens at 290498.72 tokens per second.


{'testlen': 47072, 'reflen': 47069, 'guess': [47072, 42072, 37072, 32072], 'correct': [35562, 19652, 9681, 4634]}
ratio: 1.000063736217022


PTBTokenizer tokenized 307085 tokens at 654704.30 tokens per second.
PTBTokenizer tokenized 52066 tokens at 289202.76 tokens per second.


{'testlen': 47067, 'reflen': 47128, 'guess': [47067, 42067, 37067, 32067], 'correct': [35561, 19620, 9661, 4609]}
ratio: 0.9987056526905238

Held-out Test             : COMPLETE
BLEU-1                  : 0.755436 ± 0.000854
BLEU-4                  : 0.339786 ± 0.001314
METEOR                  : 0.261338 ± 0.000621
ROUGE-L                 : 0.550405 ± 0.001158
CIDEr-D                 : 1.088168 ± 0.005240
Test open count           : 1
Seed/checkpoint selection : False
Final marker              : /kaggle/working/XLKD05_Dense_Mapped_LogitKD_held_out_test_one_time/XLKD05_Dense_Mapped_LogitKD_HELD_OUT_TEST_COMPLETE.json
